# 混合重采样策略在极度不平衡金融欺诈检测中的有效性研究
## An Empirical Research on the Efficacy of Hybrid Resampling Strategies for Extremely Imbalanced Financial Fraud Detection

**Group 3** | HU Chenjing · HE Minghao · SUN Fanhong · LI Mingzheng · LING Yupeng

---

### 实验设计概览

| 维度 | 选项 |
|------|------|
| **重采样策略** | Baseline（不处理）/ SMOTE / Tomek Links / SMOTE-Tomek |
| **分类器** | Logistic Regression / Random Forest / XGBoost |
| **主要指标** | AUPRC（Area Under PR Curve）|
| **辅助指标** | Recall / Precision / F1 / G-Mean / AUROC |
| **数据集** | Kaggle Credit Card Fraud Detection（284,807 笔，欺诈率 0.172%）|

---
## 0. 环境设置

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# 设置项目根目录（确保能 import src/）
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print('✅ 环境就绪')

---
## 1. 数据加载与探索性分析 (EDA)

In [ ]:
from src.data_loader import load_data, preprocess

# 加载数据（若本地不存在会自动下载）
df = load_data()

In [ ]:
# 基本信息
print(f'数据维度: {df.shape}')
df.head()

In [ ]:
# 类别分布可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Class'].value_counts()
axes[0].bar(['Normal (0)', 'Fraud (1)'], counts.values,
            color=['#4C72B0', '#C44E52'], edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2000, f'{v:,}\n({v/len(df)*100:.3f}%)', ha='center')
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Samples')

axes[1].pie(counts.values, labels=['Normal', 'Fraud'],
            colors=['#4C72B0', '#C44E52'], autopct='%1.3f%%',
            startangle=140, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Class Distribution (Proportion)')

plt.suptitle('Credit Card Fraud Dataset — Class Imbalance', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'\n不平衡比例: 1 : {counts[0] // counts[1]} (正常:欺诈)')

In [ ]:
# Amount 和 Time 分布对比（正常 vs 欺诈）
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['Amount', 'Time']):
    df[df['Class'] == 0][col].hist(bins=50, ax=ax, alpha=0.6,
                                   label='Normal', color='#4C72B0', density=True)
    df[df['Class'] == 1][col].hist(bins=50, ax=ax, alpha=0.8,
                                   label='Fraud', color='#C44E52', density=True)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Feature Distribution: Normal vs Fraud', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 特征相关性热力图（Top 15 特征）
plt.figure(figsize=(14, 10))
corr = df.corr()
# 按与 Class 的相关性排序，取 top 15
top_features = corr['Class'].abs().sort_values(ascending=False).head(15).index
sns.heatmap(df[top_features].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix (Top 15 by |corr with Class|)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. 数据预处理

In [ ]:
# RobustScaler 标准化 + 分层 80/20 分割
X_train, X_test, y_train, y_test = preprocess(df, verbose=True)

print(f'\nX_train: {X_train.shape} | X_test: {X_test.shape}')

---
## 3. 重采样策略对比

In [ ]:
from src.resampling import get_resampled
from collections import Counter

strategies = ['baseline', 'oversample', 'undersample', 'hybrid']
strategy_labels = {
    'baseline'   : 'Baseline (No Resampling)',
    'oversample' : 'SMOTE (Oversample)',
    'undersample': 'Tomek Links (Undersample)',
    'hybrid'     : 'SMOTE-Tomek (Hybrid)',
}

# 运行所有策略并记录分布
resampled_data = {}
for strat in strategies:
    print(f'\n=== {strategy_labels[strat]} ===')
    X_res, y_res = get_resampled(strat, X_train, y_train)
    resampled_data[strat] = (X_res, y_res)

In [ ]:
# 可视化重采样后的分布对比
counts_data = {}
for strat, (_, y_r) in resampled_data.items():
    c = Counter(y_r)
    counts_data[strategy_labels[strat]] = {'normal': c[0], 'fraud': c[1]}

strategies_list = list(counts_data.keys())
normals = [v['normal'] for v in counts_data.values()]
frauds  = [v['fraud']  for v in counts_data.values()]
x = np.arange(len(strategies_list))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w/2, normals, w, label='Normal', color='#4C72B0', alpha=0.85)
b2 = ax.bar(x + w/2, frauds,  w, label='Fraud',  color='#C44E52', alpha=0.85)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 200,
            f'{h:,.0f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(strategies_list, fontsize=10)
ax.set_ylabel('Number of Samples')
ax.set_title('Sample Distribution After Each Resampling Strategy', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. 模型训练与评估

In [ ]:
import time
from src.models     import build_model
from src.evaluation import evaluate_model

model_names = ['logistic_regression', 'random_forest', 'xgboost']

all_results  = []
all_probas   = {}   # {strategy: {model: y_proba}}

for strat in strategies:
    X_res, y_res = resampled_data[strat]
    all_probas[strat] = {}
    
    for mname in model_names:
        print(f'\n{'─'*50}')
        print(f'  {strat.upper()} × {mname}')
        print(f'{'─'*50}')
        
        model = build_model(mname)
        t0 = time.time()
        model.fit(X_res, y_res)
        train_time = time.time() - t0
        
        metrics = evaluate_model(model, X_test, y_test, verbose=True)
        
        # 保存预测概率
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)[:, 1]
        else:
            d = model.decision_function(X_test)
            y_proba = (d - d.min()) / (d.max() - d.min() + 1e-9)
        all_probas[strat][mname] = y_proba
        
        row = {
            'strategy'  : strat,
            'model'     : mname,
            'train_time': round(train_time, 3),
            **{k: v for k, v in metrics.items() if k not in ['confusion_matrix']},
        }
        all_results.append(row)

results_df = pd.DataFrame(all_results)
print('\n✅ 所有实验完成！')

---
## 5. 结果分析与可视化

In [ ]:
# ── 汇总表（按 AUPRC 降序）────────────────────────────────────────
display_cols = ['strategy', 'model', 'recall', 'precision', 'f1', 'g_mean', 'auprc', 'auroc']
summary = results_df[display_cols].sort_values('auprc', ascending=False)

print('\n实验结果汇总（按 AUPRC 降序）：')
display(summary.style
    .background_gradient(subset=['auprc', 'g_mean', 'recall'], cmap='YlGn')
    .format({
        'recall': '{:.4f}', 'precision': '{:.4f}', 'f1': '{:.4f}',
        'g_mean': '{:.4f}', 'auprc': '{:.4f}', 'auroc': '{:.4f}'
    })
)

In [ ]:
# ── 指标对比条形图 ────────────────────────────────────────────────
metrics_to_plot = ['recall', 'precision', 'f1', 'g_mean', 'auprc']
COLORS = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, len(metrics_to_plot),
                          figsize=(5 * len(metrics_to_plot), 5),
                          sharey=False)

strat_order = ['baseline', 'oversample', 'undersample', 'hybrid']
x = np.arange(len(strat_order))
bar_w = 0.8 / len(model_names)

for ax, metric in zip(axes, metrics_to_plot):
    for i, mdl in enumerate(model_names):
        sub = results_df[results_df['model'] == mdl].set_index('strategy')
        vals = [sub.loc[s, metric] if s in sub.index else 0 for s in strat_order]
        offset = (i - len(model_names)/2 + 0.5) * bar_w
        bars = ax.bar(x + offset, vals, width=bar_w,
                      label=mdl, color=COLORS[i], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom',
                    fontsize=7, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(strat_order, rotation=15, ha='right', fontsize=9)
    ax.set_ylim(0, 1.18)
    ax.set_title(metric.upper(), fontweight='bold')
    if metric == metrics_to_plot[0]:
        ax.legend(fontsize=8)

plt.suptitle('All Strategies × Models — Performance Comparison',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── PR 曲线（每种策略一张，三个模型叠加）─────────────────────────
from sklearn.metrics import precision_recall_curve, auc

fig, axes = plt.subplots(1, len(strategies), figsize=(5 * len(strategies), 5))
baseline_pr = y_test.mean()  # 随机猜测基线

for ax, strat in zip(axes, strategies):
    for i, mdl in enumerate(model_names):
        proba = all_probas[strat][mdl]
        prec, rec, _ = precision_recall_curve(y_test, proba)
        ap = auc(rec, prec)
        ax.plot(rec, prec, label=f'{mdl} (AP={ap:.3f})',
                color=COLORS[i], linewidth=2)
    ax.axhline(baseline_pr, color='gray', linestyle='--',
               linewidth=1, label=f'Random (AP={baseline_pr:.3f})')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(strategy_labels[strat], fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Precision-Recall Curves by Strategy',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC 曲线（每种策略一张）──────────────────────────────────────
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, len(strategies), figsize=(5 * len(strategies), 5))

for ax, strat in zip(axes, strategies):
    for i, mdl in enumerate(model_names):
        proba = all_probas[strat][mdl]
        fpr, tpr, _ = roc_curve(y_test, proba)
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{mdl} (AUC={roc_auc_val:.3f})',
                color=COLORS[i], linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR (Recall)')
    ax.set_title(strategy_labels[strat], fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('ROC Curves by Strategy',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap：各策略 × 模型的 AUPRC ──────────────────────────────
pivot = results_df.pivot(index='model', columns='strategy', values='auprc')
pivot = pivot[strat_order]  # 固定列顺序

plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'AUPRC'})
plt.title('AUPRC Heatmap: Strategy × Model', fontweight='bold')
plt.xlabel('Resampling Strategy'); plt.ylabel('Classifier')
plt.tight_layout()
plt.show()

In [ ]:
# ── G-Mean Heatmap ───────────────────────────────────────────────
pivot_g = results_df.pivot(index='model', columns='strategy', values='g_mean')[strat_order]

plt.figure(figsize=(8, 4))
sns.heatmap(pivot_g, annot=True, fmt='.4f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': 'G-Mean'})
plt.title('G-Mean Heatmap: Strategy × Model', fontweight='bold')
plt.xlabel('Resampling Strategy'); plt.ylabel('Classifier')
plt.tight_layout()
plt.show()

---
## 6. 保存结果

In [ ]:
import json

os.makedirs('../results', exist_ok=True)
save_cols = [c for c in results_df.columns if not c.startswith('_')]

# CSV
results_df[save_cols].to_csv('../results/experiment_results.csv',
                              index=False, float_format='%.6f')

# JSON
records = results_df[save_cols].to_dict(orient='records')
with open('../results/experiment_results.json', 'w', encoding='utf-8') as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print('✅ 结果已保存至 results/')
print('   - results/experiment_results.csv')
print('   - results/experiment_results.json')

---
## 7. 研究问题解答摘要

### RQ1: SMOTE-Tomek 混合策略是否优于单独的过采样/欠采样？
> 通过上方 AUPRC 和 G-Mean 热力图对比，可以得出结论……（填写实验结果）

### RQ2: 不同分类器在重采样后的表现差异？
> XGBoost / Random Forest / Logistic Regression 的表现对比……（填写实验结果）

### RQ3: AUPRC 和 G-Mean 是否比传统 Accuracy 更合适？
> 对比 Baseline 的 Accuracy 与 AUPRC 指标的差异……（填写实验结果）

---
*后续可添加：超参数调优（GridSearchCV）、阈值优化、SHAP 特征解释等模块*